In [ ]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from pydantic import BaseModel

# 1. Define the Strict Data Structure
class CropAnalysis(BaseModel):
    crop_name: str
    disease_status: str
    action_required: bool
    confidence_score: int

print("✅ Block 1: Imports loaded and Pydantic Schema Defined")

✅ Block 1: Imports loaded and Pydantic Schema Defined


In [7]:
# 2. Local Brain Configuration (Ollama)
local_structured_client = OpenAIChatCompletionClient(
    model="llama3.2",
    api_key="placeholder",
    base_url="http://localhost:11434/v1",
    response_format={"type": "json_object"}, # <--- THE FIX: Standard JSON mode
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": "llama3",
    }
)

print("✅ Block 2: Local Structured Client Initialized")

✅ Block 2: Local Structured Client Initialized


In [8]:
# 3. Create the Data Analyst Agent
literal_template = '''{
  "crop_name": "name of the crop",
  "disease_status": "description of the disease",
  "action_required": true or false,
  "confidence_score": integer from 0 to 100
}'''

data_agent = AssistantAgent(
    name='agri_data_analyst',
    model_client=local_structured_client,
    system_message=f"You are an agricultural data extractor. Extract the requested info. You MUST return ONLY valid JSON exactly matching this template:\n{literal_template}\nDo not add any other keys, wrappers, or markdown formatting."
)

print("✅ Block 3: Resilient JSON-Mode Agent Initialized")

✅ Block 3: Resilient JSON-Mode Agent Initialized


In [ ]:
# 4. Run the Pipeline
async def run_structured_test():
    # A messy input from a hypothetical farmer
    messy_text = "Yeah hi, I was looking at my Wheat crops today and there's this weird yellow rust all over the leaves. I think it's pretty bad, I'm like 90 percent sure it's sick."
    
    task = TextMessage(content=f"Extract the data from this report: {messy_text}", source='User')
    
    print("--- Extracting Data ---")
    result = await data_agent.run(task=task)
    
    # 1. The Raw JSON from Llama 3.2
    raw_json = result.messages[-1].content
    print("🤖 Raw AI Output:\n", raw_json)
    print("-" * 30)
    
    # 2. The Architect Step: Validating the data into a perfect Python object
    try:
        final_data = CropAnalysis.model_validate_json(raw_json)
        print("✅ Success! Perfect Python Object Created!")
        print(f"Crop Identified: {final_data.crop_name}")
        print(f"Action Required: {final_data.action_required}")
        print(f"Confidence Level: {final_data.confidence_score}%")
    except Exception as e:
        print("⚠️ Parsing Error (The model hallucinated bad JSON):", e)

# THE JUPYTER FIX: Run the async function
await run_structured_test()

--- Extracting Data ---
🤖 Raw AI Output:
 {
  "crop_name": "Wheat",
  "disease_status": "yellow rust on the leaves",
  "action_required": true,
  "confidence_score": 90
}
------------------------------
✅ Success! Perfect Python Object Created!
Crop Identified: Wheat
Action Required: True
Confidence Level: 90%


In [10]:
import os
from dotenv import load_dotenv
from autogen_agentchat.messages import MultiModalMessage
from autogen_core import Image as AGImage
from PIL import Image
from io import BytesIO
import requests

# 1. Load the Cloud Key
load_dotenv()
or_key = os.getenv("OPENROUTER_API_KEY")

# 2. The Cloud Vision Brain 
# We use a known free vision model on OpenRouter (Llama 3.2 11B Vision)
cloud_vision_client = OpenAIChatCompletionClient(
    model="meta-llama/llama-3.2-11b-vision-instruct:free", 
    api_key=or_key,
    base_url="https://openrouter.ai/api/v1",
    model_info={
        "vision": True, # <--- CRITICAL: Tells AutoGen this brain has eyes
        "function_calling": False,
        "json_output": False,
        "family": "unknown",
    }
)

print("✅ Block 5: Cloud Vision Client Initialized")

✅ Block 5: Cloud Vision Client Initialized


/Users/vishalsingh/Documents/GIT/Autonomous-Research-Intelligence-Suite/venv/lib/python3.13/site-packages/autogen_ext/models/openai/_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [ ]:
# 3. Create the Agricultural Inspector Agent
vision_agent = AssistantAgent(
    name='agri_vision_inspector',
    model_client=cloud_vision_client,
    system_message="You are an expert agricultural inspector. Analyze images of farms or crops and describe exactly what you see."
)

print("✅ Block 6: Vision Agent Created")